# Company Information

This notebook presents stored company metadata, point-in-time market data, and
balance-sheet-derived book value metrics without live data requests.


In [1]:
from src.setup import *

## Step 1: Data Categories

The following information is collected:

- Stored company metadata (name, country, sector, industry, currency, website)
- Share price and market capitalization at the fixed market snapshot date
- Book value per share and price-to-book derived from stored snapshot data


In [2]:

def company_information(
    company_profile,
    quarterly_balance_sheet,
    price,
    market_cap,
    price_date,
):
    """
    Collect company information from the stored project snapshots.

    Parameters
    ----------
    company_profile : dict
        Stored company profile fields.
    quarterly_balance_sheet : pandas.DataFrame
        Quarterly balance sheet snapshot with the latest period first.
    price : float
        Share price as of the market snapshot date.
    market_cap : float
        Market capitalization as of the market snapshot date.
    price_date : pandas.Timestamp
        Date of the stored market snapshot.

    Returns
    -------
    pd.DataFrame
        Company metadata and key descriptive metrics.
    """
    book_equity = quarterly_balance_sheet.loc["StockholdersEquity"].iloc[0]
    book_shares = quarterly_balance_sheet.loc["OrdinarySharesNumber"].iloc[0]
    balance_sheet_date = quarterly_balance_sheet.columns[0].date()
    book_value_per_share = (
        book_equity / book_shares
        if book_shares != 0
        else np.nan
    )
    price_to_book = (
        price / book_value_per_share
        if book_value_per_share != 0
        else np.nan
    )

    data = {
        "Company Name": company_profile["longName"],
        "Country": company_profile["country"],
        "Sector": company_profile["sector"],
        "Industry": company_profile["industry"],
        "Currency": company_profile["currency"],
        "Website": company_profile["website"],
        f"Share Price as of {pd.Timestamp(price_date).date()}": price,
        f"Market Capitalization as of {pd.Timestamp(price_date).date()}": market_cap,
        f"Book Value per Share as of {balance_sheet_date}": book_value_per_share,
        f"Price-to-Book Ratio as of {pd.Timestamp(price_date).date()}": price_to_book,
        "Business Summary": company_profile["longBusinessSummary"],
    }

    df_info = pd.DataFrame.from_dict(data, orient="index", columns=["Value"])
    return df_info


## Step 2: Run Company Information Extraction


In [3]:
df_info = company_information(
    company_profile=company_profile,
    quarterly_balance_sheet=quarterly_balance_sheet,
    price=price,
    market_cap=market_cap,
    price_date=price_date,
)

## Step 3: Display Results


In [4]:
with pd.option_context("display.float_format", "{:,.2f}".format):
    display(df_info)

,Value
Company Name,Microsoft Corporation
Country,United States
Sector,Technology
Industry,Software - Infrastructure
Currency,USD
Website,https://www.microsoft.com
Share Price as of 2025-12-26,487.71
Market Capitalization as of 2025-12-26,"3,624,844,928,067.00"
Book Value per Share as of 2025-09-30,48.85
Price-to-Book Ratio as of 2025-12-26,9.98
